# RAG Policy Comparator
Upload **two policy documents** (PDF / DOCX / TXT) and ask questions about how they differ. Answers are generated **only from retrieved text**, with citations.

**How it works:** parse → section-aware chunking → embeddings → retrieve top-k chunks **from each document separately** (so one doc can't crowd out the other) → a free open-source LLM (runs inside Colab, no API key) compares using only that context.

**Before you start:** Runtime → Change runtime type → **T4 GPU** (free). CPU works but is slow.

**Run order:** 1 install → 2 load local model → 3 documents → 4 RAG engine → 5 ask → 6 optional test suite.

### 1. Install

In [ ]:
!pip install -q pypdf python-docx sentence-transformers transformers accelerate scikit-learn reportlab

### 2. Load a free local LLM
No API key needed. Downloads **Qwen2.5-3B-Instruct** (~6 GB, one-time, ~2-3 min) on a GPU runtime, or the smaller 1.5B model on CPU. Weights stay inside your Colab session.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

GPU = torch.cuda.is_available()
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct" if GPU else "Qwen/Qwen2.5-1.5B-Instruct"
print("GPU:", GPU, "| model:", MODEL_ID)

tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if GPU else torch.float32,
    device_map="auto" if GPU else None,
)
model.eval()

def generate(system, user, max_new_tokens=450):
    msgs = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

print(generate("You are helpful.", "Say OK."))

### 3. Documents
**Option A (default):** generate 3 test policies with known, deliberate differences.  
**Option B:** upload your own two files (set `USE_TEST_DOCS = False`).

In [ ]:
import os
USE_TEST_DOCS = True      # False -> upload your own two files
OUT = "test_policies"
from reportlab.lib.pagesizes import A4
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet

DOCS = {
"01_Acme_Leave_Remote_Policy_2023": ("Acme Corp - Leave & Remote Work Policy (Version 2023)", [
 ("1. Purpose", "This policy defines leave entitlements and remote work rules for all full-time employees of Acme Corp. It is effective from 1 January 2023 and supersedes all earlier circulars."),
 ("2. Working Hours", "Standard working hours are 9:00 AM to 6:00 PM, Monday to Friday, with a one-hour lunch break. Core collaboration hours are 11:00 AM to 4:00 PM."),
 ("3. Annual Leave", "Every full-time employee is entitled to 18 days of paid annual leave per calendar year. Up to 5 unused days may be carried forward to the next year; any balance above 5 days lapses on 31 March. Leave must be requested at least 7 days in advance for periods longer than 2 days."),
 ("4. Sick Leave", "Employees receive 8 days of paid sick leave per year. A medical certificate is required if the absence exceeds 2 consecutive days. Unused sick leave cannot be carried forward or encashed."),
 ("5. Maternity and Paternity Leave", "Maternity leave is 12 weeks of fully paid leave. Paternity leave is 5 working days, to be taken within 30 days of the child's birth."),
 ("6. Remote Work", "Employees may work remotely up to 2 days per week with prior approval from their line manager. Remote work is not permitted during the first 3 months of employment (probation). The company provides no home-office allowance."),
 ("7. Leave Encashment on Exit", "On separation, employees may encash up to 10 days of unused annual leave at the basic daily rate. Encashment is paid with the final settlement."),
 ("8. Notice Period", "The standard notice period for resignation is 30 days for all grades. Employees on probation must serve 15 days."),
 ("9. Public Holidays", "Acme observes 12 public holidays per year, published on the intranet in December for the next calendar year."),
]),
"02_Acme_Leave_Remote_Policy_2025": ("Acme Corp - Leave & Remote Work Policy (Version 2025)", [
 ("1. Purpose", "This policy defines leave entitlements and remote work rules for all full-time employees of Acme Corp. It is effective from 1 January 2025 and replaces the 2023 version."),
 ("2. Working Hours", "Standard working hours are 9:00 AM to 6:00 PM, Monday to Friday, with a one-hour lunch break. Core collaboration hours are 11:00 AM to 4:00 PM."),
 ("3. Annual Leave", "Every full-time employee is entitled to 24 days of paid annual leave per calendar year. Up to 10 unused days may be carried forward to the next year; any balance above 10 days lapses on 30 June. Leave must be requested at least 5 days in advance for periods longer than 3 days."),
 ("4. Sick Leave", "Employees receive 12 days of paid sick leave per year. A medical certificate is required if the absence exceeds 3 consecutive days. Unused sick leave cannot be carried forward or encashed."),
 ("5. Maternity and Paternity Leave", "Maternity leave is 26 weeks of fully paid leave. Paternity leave is 15 working days, which may be taken in up to two blocks within 6 months of the child's birth."),
 ("6. Mental Wellness Days", "In addition to sick leave, each employee receives 2 paid mental wellness days per year. No medical certificate or reason needs to be provided. These days cannot be carried forward."),
 ("7. Remote Work", "Employees may work remotely up to 3 days per week with prior approval from their line manager. Remote work is permitted after the first month of employment. A one-time home-office allowance of INR 15,000 is provided after probation."),
 ("8. Leave Encashment on Exit", "Encashment of unused annual leave on exit is no longer permitted. Unused leave lapses on the last working day."),
 ("9. Notice Period", "The standard notice period is 60 days for managers and above and 30 days for all other grades. Employees on probation must serve 15 days."),
 ("10. Public Holidays", "Acme observes 12 public holidays per year, published on the intranet in December for the next calendar year."),
]),
"03_NovaTech_Leave_Policy": ("NovaTech Solutions - Employee Leave & Flexible Work Policy", [
 ("1. Scope", "This policy applies to all permanent employees of NovaTech Solutions and is effective from 1 April 2024."),
 ("2. Paid Time Off", "Employees receive 20 days of paid time off (PTO) per year. PTO follows a use-it-or-lose-it rule: no days may be carried forward, and unused PTO is not paid out on exit. Requests need 3 days' notice."),
 ("3. Sick Leave", "Employees receive 10 days of paid sick leave annually. No medical certificate is required for absences up to 5 consecutive days; beyond that, a certificate must be submitted to HR."),
 ("4. Parental Leave", "Primary caregivers receive 16 weeks of paid parental leave, and secondary caregivers receive 10 working days. Leave is available irrespective of gender."),
 ("5. Flexible Work", "Employees may work remotely up to 4 days per week. Team leads may require in-office presence on specific days with 48 hours' notice. NovaTech provides a monthly internet reimbursement of INR 1,000."),
 ("6. Sabbatical", "Employees with at least 5 years of continuous service may apply for an unpaid sabbatical of 1 to 3 months. Approval is at the discretion of the department head."),
 ("7. Notice Period", "The notice period is 45 days for all employees, reduced to 30 days during the first year of service."),
 ("8. Public Holidays", "NovaTech observes 10 public holidays plus 2 floating holidays that employees may choose from an approved list."),
]),
}


os.makedirs(OUT, exist_ok=True)
_styles = getSampleStyleSheet()
for fname, (title, sections) in DOCS.items():
    d = SimpleDocTemplate(f"{OUT}/{fname}.pdf", pagesize=A4, title=title)
    story = [Paragraph(title, _styles["Title"]), Spacer(1, 12)]
    for head, body in sections:
        story += [Paragraph(head, _styles["Heading2"]), Paragraph(body, _styles["Normal"]), Spacer(1, 8)]
    d.build(story)
print("Test PDFs:", sorted(os.listdir(OUT)))

# ---- choose which two documents to compare ----
if USE_TEST_DOCS:
    PATH_A = f"{OUT}/01_Acme_Leave_Remote_Policy_2023.pdf"
    PATH_B = f"{OUT}/02_Acme_Leave_Remote_Policy_2025.pdf"
    # Try other pairs too, e.g. PATH_B = f"{OUT}/03_NovaTech_Leave_Policy.pdf"
else:
    from google.colab import files
    up = files.upload()
    names = list(up.keys())
    assert len(names) >= 2, "Please upload two files"
    PATH_A, PATH_B = names[0], names[1]
print("Comparing:", PATH_A, "<->", PATH_B)

### 4. RAG engine

In [ ]:
import re, os, numpy as np
from pypdf import PdfReader

# ---------- 1. Load ----------
def load_text(path):
    ext = os.path.splitext(path)[1].lower()
    if ext == ".pdf":
        return "\n".join((p.extract_text() or "") for p in PdfReader(path).pages)
    if ext == ".docx":
        import docx
        return "\n".join(p.text for p in docx.Document(path).paragraphs)
    return open(path, encoding="utf-8", errors="ignore").read()

# ---------- 2. Chunk (section-aware) ----------
HEADING = re.compile(r"^\s*(\d+(\.\d+)*\.?|[A-Z][A-Za-z]*\.|Article\s+\d+|Section\s+\d+)\s+\S.{0,80}$")

def chunk_text(text, max_chars=900, overlap=150):
    """Split on headings; long sections are split by sentences with overlap. Each chunk keeps its heading."""
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    sections, head, buf = [], "Preamble", []
    for l in lines:
        if HEADING.match(l) and len(l) < 90 and not l.endswith("."):
            if buf: sections.append((head, " ".join(buf)))
            head, buf = l, []
        else:
            buf.append(l)
    if buf: sections.append((head, " ".join(buf)))

    chunks = []
    for head, body in sections:
        if len(body) <= max_chars:
            chunks.append({"section": head, "text": f"{head}: {body}"})
            continue
        sents, cur = re.split(r"(?<=[.!?])\s+", body), ""
        for s in sents:
            if len(cur) + len(s) > max_chars and cur:
                chunks.append({"section": head, "text": f"{head}: {cur.strip()}"})
                cur = cur[-overlap:] + " " + s
            else:
                cur += " " + s
        if cur.strip(): chunks.append({"section": head, "text": f"{head}: {cur.strip()}"})
    return chunks

# ---------- 3. Embed ----------
class Embedder:
    def __init__(self):
        try:
            from sentence_transformers import SentenceTransformer
            self.m = SentenceTransformer("all-MiniLM-L6-v2"); self.kind = "sentence-transformers"
        except Exception as e:
            print("sentence-transformers unavailable, using TF-IDF fallback:", e)
            self.m, self.kind = None, "tfidf"
    def fit(self, corpus):
        if self.kind == "tfidf":
            from sklearn.feature_extraction.text import TfidfVectorizer
            self.v = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True).fit(corpus)
    def encode(self, texts):
        if self.kind == "tfidf":
            from sklearn.preprocessing import normalize
            return normalize(self.v.transform(texts)).toarray()
        return self.m.encode(texts, normalize_embeddings=True)

# ---------- 4. Index ----------
class PolicyIndex:
    def __init__(self, path_a, path_b):
        self.names = {"A": os.path.basename(path_a), "B": os.path.basename(path_b)}
        self.chunks = {"A": chunk_text(load_text(path_a)), "B": chunk_text(load_text(path_b))}
        self.emb = Embedder()
        self.emb.fit([c["text"] for k in "AB" for c in self.chunks[k]])
        self.vecs = {k: self.emb.encode([c["text"] for c in self.chunks[k]]) for k in "AB"}
        print(f"Indexed A: {len(self.chunks['A'])} chunks | B: {len(self.chunks['B'])} chunks | embedder: {self.emb.kind}")

    def retrieve(self, query, k=4):
        """Top-k PER DOCUMENT - guarantees both sides are represented."""
        q = self.emb.encode([query])[0]
        out = {}
        for d in "AB":
            sims = self.vecs[d] @ q
            top = np.argsort(-sims)[:k]
            out[d] = [(self.chunks[d][i], float(sims[i])) for i in top]
        return out

# ---------- 5. Generate ----------
SYSTEM = """You compare two policy documents. Use ONLY the context excerpts given. Never use outside knowledge.
Answer in exactly this format:
**Document A:** what A says (with exact numbers/conditions) [A: section name]
**Document B:** what B says (with exact numbers/conditions) [B: section name]
**Difference:** one or two sentences stating what changed, or "No difference."
If a topic is missing from one document, write "Not mentioned in this document." for that document.
Do not add anything else."""

def build_context(idx, hits):
    parts = []
    for d in "AB":
        parts.append(f"=== Document {d}: {idx.names[d]} ===")
        for c, s in hits[d]:
            parts.append(f"[{d}: {c['section']}]\n{c['text']}")
    return "\n\n".join(parts)

def ask(idx, question, k=3, show_chunks=False):
    hits = idx.retrieve(question, k)
    if show_chunks:
        for d in "AB":
            print(f"\n--- Retrieved from {d} ({idx.names[d]}) ---")
            for c, s in hits[d]:
                print(f"  [{s:.2f}] {c['text'][:160]}...")
    return generate(SYSTEM, f"CONTEXT:\n{build_context(idx, hits)}\n\nQUESTION: {question}")

# ---------- 6. No-LLM side-by-side diff (instant, deterministic) ----------
import difflib, html
from IPython.display import HTML, display

def diff_view(idx, question, k=1):
    """Word-level highlighted diff of the best-matching passage from each doc. Red = only in A, green = only in B."""
    hits = idx.retrieve(question, k)
    for (ca, _), (cb, _) in zip(hits["A"], hits["B"]):
        a, b = ca["text"].split(), cb["text"].split()
        ha, hb = [], []
        for tag, i1, i2, j1, j2 in difflib.SequenceMatcher(None, a, b).get_opcodes():
            if tag == "equal":
                ha.append(html.escape(" ".join(a[i1:i2]))); hb.append(html.escape(" ".join(b[j1:j2])))
            else:
                if i2 > i1: ha.append(f"<span style='background:#ffc9c9;color:#000'>{html.escape(' '.join(a[i1:i2]))}</span>")
                if j2 > j1: hb.append(f"<span style='background:#c9f5c9;color:#000'>{html.escape(' '.join(b[j1:j2]))}</span>")
        display(HTML(f"<table style='width:100%;table-layout:fixed;border-collapse:collapse'><tr>"
                     f"<td style='vertical-align:top;padding:8px;border:1px solid #888'><b>A: {idx.names['A']}</b><br>{' '.join(ha)}</td>"
                     f"<td style='vertical-align:top;padding:8px;border:1px solid #888'><b>B: {idx.names['B']}</b><br>{' '.join(hb)}</td></tr></table>"))

idx = PolicyIndex(PATH_A, PATH_B)

### 5. Ask questions
Try: *What are the differences in annual leave?* · *Which document gives more sick leave?* · *What is only in one of the two documents?*

In [ ]:
from IPython.display import Markdown, display
q = "What changed in annual leave and carry-forward rules between the two documents?"
display(Markdown(ask(idx, q, show_chunks=True)))

**Instant no-LLM view** - highlights exactly which words differ in the best-matching passage:

In [ ]:
diff_view(idx, "annual leave carry forward")

In [ ]:
# Interactive loop - type a question, press Enter on an empty line to stop
while True:
    q = input("\nYour question (blank to stop): ").strip()
    if not q: break
    display(Markdown(ask(idx, q)))

### 6. Test suite (optional)
Checks **retrieval** (right section returned for each doc?) and **answers** (do they contain the expected facts?). Small local models can miss a detail - "CHECK" means read that answer yourself. Written for the default pair: **Acme 2023 vs Acme 2025**.

In [ ]:
TESTS = [
 # (question, section keyword that MUST be retrieved, facts expected in the answer)
 ("How many annual leave days does each document give?",          "Annual Leave",  ["18", "24"]),
 ("What is the carry-forward limit for unused leave?",            "Annual Leave",  ["5", "10"]),
 ("When is a medical certificate needed for sick leave?",         "Sick Leave",    ["2", "3"]),
 ("How long is maternity leave in each?",                         "Maternity",     ["12", "26"]),
 ("How many remote work days per week are allowed?",              "Remote Work",   ["2", "3"]),
 ("Can employees encash unused leave when they leave?",           "Encashment",    ["10", "no longer"]),
 ("Is there a mental wellness day benefit?",                      "Mental",        ["2", "not mentioned"]),
 ("What is the notice period?",                                   "Notice Period", ["30", "60"]),
 ("Are working hours different?",                                 "Working Hours", ["9"]),
]
ok_ret = ok_ans = 0
for q, sec, facts in TESTS:
    hits = idx.retrieve(q, k=3)
    # Mental wellness exists only in B, so require it from B; others from both
    need = ["B"] if sec == "Mental" else ["A", "B"]
    r = all(any(sec.lower() in c["section"].lower() for c, _ in hits[d]) for d in need)
    ok_ret += r
    line = f"{'PASS' if r else 'FAIL'} retrieval | {q}"
    a = ask(idx, q).lower()
    f = all(x.lower() in a for x in facts)
    ok_ans += f
    line += f"  || {'PASS' if f else 'CHECK'} answer"
    print(line)
print(f"\nRetrieval: {ok_ret}/{len(TESTS)} | Answers: {ok_ans}/{len(TESTS)}")